In [1]:

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # Suppress pandas FutureWarnings globally

import pandas as pd
import numpy as np
import sys
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import os
import pickle

# Add structured_products library to PYTHONPATH
sys.path.append('C:/ActData/Python/files/OOI/structured-products-main/')

# Import custom modules
from structured_products import clo, sharepoint
from dealpivot import email_deal
import dashboard

# --- Caching utility ---
def load_or_cache(path, loader_func):
    # DEPRECATED: Use safe_load_or_cache instead
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    obj = loader_func()
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    return obj

# --- Utility: Safe cache load (handles pickle incompatibility) ---
def safe_load_or_cache(path, loader_func):
    import pickle
    import os
    try:
        if os.path.exists(path):
            with open(path, 'rb') as f:
                return pickle.load(f)
    except Exception as e:
        # Only print warning if not a FutureWarning
        if not isinstance(e, FutureWarning):
            print(f"Warning: Cache load failed for {path} due to: {e}. Rebuilding cache...")
        try:
            os.remove(path)
        except Exception:
            pass
    obj = loader_func()
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    return obj

# Load CLO holdings with caching
holdings = safe_load_or_cache('holdings_cache.pkl', lambda: clo.holdings(type='portfolio'))
as_of_date = pd.to_datetime(holdings.iloc[0]['As_at_Date']).date().strftime('%B %d, %Y')

# Print summary
print(f'data as of {as_of_date}')
print(f'{holdings.shape[0]} positions loaded')
print(f"{holdings['Primary Security ID'].nunique()} unique securities")
print(f"${holdings['GAAP BV'].sum():,.2f} total GAAP book value")

# Load supplemental data from SharePoint with caching
sp = sharepoint.ABSStrucCredit()
deals = safe_load_or_cache('deals_cache.pkl', lambda: pd.DataFrame(sp.clo_deals()))
managers = safe_load_or_cache('managers_cache.pkl', lambda: pd.DataFrame(sp.clo_managers().reset_index().rename(columns={'ID': 'ManagerId'})))

# Prepare and merge metadata (optimize types)
for col in ['DealName', 'CollateralType', 'BloombergDealName']:
    if col in deals:
        deals[col] = deals[col].astype('category')
for col in ['ManagerId','ManagerName', 'ShortName', 'UltimateParent']:
    if col in managers:
        managers[col] = managers[col].astype('category')

deals = deals[['DealName', 'CollateralType', 'BloombergDealName', 'ManagerId']]
managers = managers[['ManagerId','ManagerName', 'ShortName', 'UltimateParent']]
deals = deals.merge(managers, how='left', left_on='ManagerId', right_on='ManagerId')
holdings = holdings.merge(deals, how='left', left_on='Creditor Name', right_on='BloombergDealName')

print(f"{holdings['BloombergDealName'].nunique()} unique deals")

# Dashboard generation moved to its own cell below

data as of March 20, 2026
1318 positions loaded
315 unique securities
$8,510,985,157.73 total GAAP book value
connected to abs-struc-credit
147 unique deals


In [5]:
# Group by 'Primary Security ID' and sum 'GAAP BV', then copy to clipboard without column headers
holdings.groupby('Primary Security ID')[['GAAP BV']].sum().to_clipboard(header=False)

Primary Security ID
00177GAY7    35000000.00
00177GBC4    12000000.00
00177GBE0     7500000.00
00177RAJ6    72000000.00
00177RAN7    24000000.00
                ...     
97719VAC3    20000000.00
97720BAA8    91000000.00
97720BAC4    24663903.28
97720CAA6    13500000.00
97720CAC2     7500000.00
Name: GAAP BV, Length: 315, dtype: float64